# AI Research Paper Intelligence System
Semantic search over ML research paper abstracts, with automatic summarization, keyword extraction, and domain-specific entity classification.

## 1. Load Dataset
Download the ArXiv ML papers dataset from Hugging Face.


In [1]:
!pip install datasets


In [2]:
from datasets import load_dataset


In [3]:
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')


In [4]:
print(dataset)


Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [5]:
dataset[0]


{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

## 2. Data Cleaning & Preparation
Keep only the relevant columns, cap dataset size, and merge title + abstract into a single text field for embedding.


In [6]:
import pandas as pd


In [7]:
df=pd.DataFrame(dataset)
df


,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [8]:
df.columns


Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [9]:
df=df[['title','abstract']]
df


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [10]:
df.shape


(117592, 2)

In [11]:
df=df.head(15000)


##11700 for each word embedding  117000*400=million
##float ka byte 4

In [12]:
df.shape


(15000, 2)

In [13]:
df.isnull().sum()


,0
title,0
abstract,0


In [14]:
df["paper_text"]=df["title"]+" "+df["abstract"]


/tmp/ipykernel_477/155913167.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["title"]+" "+df["abstract"] ## not only titke but also abstract it should  check


In [15]:
df[["paper_text"]].head()


,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [16]:
type(df[["paper_text"]])


pandas.core.frame.DataFrame

In [17]:
df[["paper_text"]].head()


,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [18]:
print(df["paper_text"].iloc[0])


Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

## 3. Generate Sentence Embeddings
Convert paper text into 384-dimensional semantic vectors using a Sentence-Transformer model.


In [19]:
from sentence_transformers import SentenceTransformer


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)


In [21]:
print(type(model))


<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [22]:
sample_text=df['paper_text'].iloc[0]


In [23]:
sample_text


'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

In [24]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()


/tmp/ipykernel_477/2978923763.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False) ##convert \n to space
/tmp/ipykernel_477/2978923763.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.strip() ## starting n end spaces remove


In [25]:
sample_text=df["paper_text"].iloc[0]
sample_text


'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [26]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)


<class 'numpy.ndarray'>
(384,)


In [27]:
embedding[:56]


array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474,
        0.05692544,  0.0201578 ,  0.0258311 ,  0.0321703 ,  0.10513764,
       -0.09676763,  0.02700802, -0.0234509 , -0.04549678, -0.01013699,
       -0.01794855, -0.04814427,  0.01077652, -0.03759069,  0.01943481,
        0.03715189,  0.02967844,  0.04330941,  0.04373213,  0.03704866,
       -0.00182594,  0.00455183, -0.00799067,  0.03037368, -0.014378  ,
        0.03795147,  0.0595916 , -0.02583356, -0.06521576,  0.05900268,
       -0.02107134,  0.07359422, -0.05720106,  0.00294061,  0.00767515,
       -0.0333416 ], dtype=float32)

In [28]:

sample_embedding=model.encode(df["paper_text"].head(5).to_list())


In [29]:
print(sample_embedding.shape)
print(sample_embedding.shape)


(5, 384)


## 4. Testing Similarity (Small Sample)
Quick sanity check: compare a few sample embeddings using cosine similarity before scaling up.


In [30]:
from sklearn.metrics.pairwise import cosine_similarity


##Y reshape data??
##getting 2D as output

In [31]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)


[[1.0000001]]


In [32]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)


[[0.36625272]]


In [33]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)


[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


##**Generate** **full** **Embedding**

In [ ]:
import os
import numpy as np

if os.path.exists("paper_embedding.npy"):
  print("Loading saved embedding")
  embedding = np.load("paper_embedding.npy")
else:
  print("Generating embedding")
  embedding = model.encode(df["paper_text"].tolist(), batch_size=32, show_progress_bar=True)
  np.save("paper_embedding.npy", embedding)
print("Embeddings saved successfully")


In [ ]:

print(embedding.shape)
print(type(embedding))


In [ ]:
embedding.dtype


## 5. Build FAISS Vector Index
Store all paper embeddings in a FAISS index for fast nearest-neighbor semantic search.


In [ ]:
!pip install faiss-cpu


In [ ]:
import faiss


In [ ]:
import os
if os.path.exists("paper_faiss.index"):
  print("Loading existing Faiss index")
  index=faiss.read_index("paper_faiss.index")
else:
  print("Creating new FAISS index")
  faiss.normalize_L2(embedding)
  index=faiss.IndexFlatL2(384)
  index.add(embedding)
  faiss.write_index(index,"paper_faiss.index")
  print("FAISS index saved successfully!")


In [ ]:
faiss.normalize_L2(embedding)


In [ ]:
index=faiss.IndexFlatL2(384)


In [ ]:
index.add(embedding)


In [ ]:
print(index.ntotal)


## 6. Semantic Search
Encode a user query and retrieve the most similar papers from the FAISS index.


In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape


In [ ]:
faiss.normalize_L2(query_embedding)


In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)


In [ ]:
print(df.iloc[10466]["abstract"])


In [ ]:
print(df.iloc[11873]["title"])


In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(embedding)
  D,I=index.search(query_embedding,k)
  return D,I


In [ ]:
D,I=search_paper("deep learning for medical image analysis")
print(D)
print(I)


In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()


In [ ]:
search_paper("deep learning for medical image analysis")


## 7. Summarization (BART)
Condense each retrieved abstract into a short, readable summary.


In [ ]:
!pip install -q transformers datasets


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

def summarizer(text, max_length=120, min_length=40, do_sample=False):
    inputs = _tokenizer([text], max_length=1024, truncation=True, return_tensors="pt")
    summary_ids = _model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        do_sample=do_sample
    )
    decoded = _tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return [{"summary_text": decoded}]


In [ ]:
type(summarizer)


In [ ]:
import pandas as pd
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)


In [ ]:
type(summary)


In [ ]:
type(summary[0])


In [ ]:
summary[0]["summary_text"]


In [ ]:
for score,idx in zip(D[0],I[0]):
   print("Similarity score",score)
   print("Title",df.iloc[idx]["title"])
   print("Abstract",df.iloc[idx]["abstract"][:500])

   summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
   print(summary)
   print(summary[0]["summary_text"])
   print()


In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print(summary)
    print(summary[0]["summary_text"])
    print()


In [ ]:
search_paper("deep learning for medical image analysis",k=5)


## 8. Keyword Extraction (KeyBERT)
Pull out the most representative keywords/phrases from each abstract.


In [ ]:
pip install keybert==0.8.5


In [ ]:
from keybert import KeyBERT


In [ ]:
kw_model = KeyBERT(model='all-MiniLm-L6-v2')


In [ ]:
type(kw_model)


In [ ]:
text=df.iloc[10466]["abstract"]
keywords = kw_model.extract_keywords(text)


In [ ]:
print(keywords)


In [ ]:
print(type(keywords[0]))


In [ ]:
keywords=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words="english")


In [ ]:
print(keywords)


In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print(summary)
    print(summary[0]["summary_text"])
    print()

    keywords=kw_model.extract_keywords(df.iloc[idx]["abstract"], keyphrase_ngram_range=(1, 3), stop_words="english")
    print(keywords)
    for k,s in keywords:
      print(k,s)
    print()


## NER - Domain Specific Entity Classification
Classifies important ML/NLP keywords found in text into categories such as MODEL, FRAMEWORK, LANGUAGE, ALGORITHM, DATASET, METRIC.


In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm


In [ ]:
import spacy
ner_model = spacy.load("en_core_web_sm")


In [ ]:
ENTITY_GAZETTEER = {
    "MODEL": [
        "deep learning", "machine learning", "neural network", "artificial intelligence",
        "transformer", "attention mechanism", "bert", "gpt", "gpt-2", "gpt-3", "gpt-4",
        "cnn", "convolutional neural network", "rnn", "recurrent neural network", "lstm",
        "gru", "resnet", "vgg", "yolo", "u-net", "gan", "generative adversarial network",
        "vae", "autoencoder", "bart", "t5", "roberta", "diffusion model", "reinforcement learning"
    ],
    "FRAMEWORK": [
        "tensorflow", "pytorch", "keras", "fastapi", "flask", "django", "spring", "spring boot",
        "scikit-learn", "sklearn", "huggingface", "hugging face", "langchain", "react", "angular",
        "vue", "node.js", "express", "next.js", "streamlit"
    ],
    "LANGUAGE": [
        "python", "java", "c++", "javascript", "typescript", "go", "golang", "rust", "r", "kotlin",
        "swift", "sql", "c#", "php", "scala"
    ],
    "ALGORITHM": [
        "gradient descent", "backpropagation", "sgd", "stochastic gradient descent", "adam optimizer",
        "random forest", "decision tree", "svm", "support vector machine", "k-means", "knn",
        "naive bayes", "linear regression", "logistic regression", "xgboost"
    ],
    "DATASET": [
        "imagenet", "mnist", "coco", "cifar-10", "cifar-100", "squad", "glue", "wikitext", "arxiv"
    ],
    "METRIC": [
        "accuracy", "precision", "recall", "f1 score", "bleu", "rouge", "perplexity", "auc", "map",
        "mean average precision", "rmse", "mae"
    ],
    "LIBRARY": [
        "numpy", "pandas", "matplotlib", "opencv", "nltk", "spacy", "seaborn", "faiss", "keybert"
    ],
    "HARDWARE": [
        "gpu", "tpu", "cuda", "cpu"
    ]
}


In [ ]:
from spacy.matcher import PhraseMatcher

entity_matcher = PhraseMatcher(ner_model.vocab, attr="LOWER")
for label, terms in ENTITY_GAZETTEER.items():
    patterns = [ner_model.make_doc(term) for term in terms]
    entity_matcher.add(label, patterns)

def classify_entities(text):
    doc = ner_model(text)
    matches = entity_matcher(doc)
    results = []
    seen = set()
    for match_id, start, end in matches:
        label = ner_model.vocab.strings[match_id]
        span_text = doc[start:end].text
        key = (span_text.lower(), label)
        if key in seen:
            continue
        seen.add(key)
        results.append((span_text, label))
    return results


In [ ]:
query = "fastapi framework for building apis in python"
entities = classify_entities(query)
for ent_text, label in entities:
    print(ent_text, "->", label)


In [ ]:
def search_paper_full(query, k=5):
  print("Query Entities:")
  query_entities = classify_entities(query)
  if query_entities:
    for ent_text, label in query_entities:
      print(" -", ent_text, "(", label, ")")
  else:
    print(" - none found")
  print()

  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D, I = index.search(query_embedding, k)

  for score, idx in zip(D[0], I[0]):
    abstract = df.iloc[idx]["abstract"]

    print("Similarity score:", score)
    print("Title:", df.iloc[idx]["title"])
    print("Abstract:", abstract[:500])
    print()

    summary = summarizer(abstract, max_length=120, min_length=40, do_sample=False)
    print("Summary:", summary[0]["summary_text"])
    print()

    keywords = kw_model.extract_keywords(abstract, keyphrase_ngram_range=(1, 3), stop_words="english")
    print("Keywords:")
    for kw, s in keywords:
      print(" -", kw, round(s, 3))
    print()

    entities = classify_entities(abstract)
    print("Entities:")
    if entities:
      for ent_text, label in entities:
        print(" -", ent_text, "(", label, ")")
    else:
      print(" - none found")
    print("=" * 80)


In [ ]:
search_paper_full("fastapi framework for building apis in python", k=3)
